# Notebook 00 — Config & Utils
## Few-Shot Learning Based HAR: Generalization to Personalization

**Purpose:** This is the foundation notebook. It must be run before ANY other notebook.  
It defines:
- All paths (datasets, outputs, checkpoints)
- Global hyperparameters
- Dataset-specific configs (sensors, sampling rates, activity labels)
- Preprocessing utilities (resampling, windowing)
- User-level train/val/test splitting
- Episode sampler (N-way K-shot)
- Metrics (accuracy, macro F1, personalization gain)
- Save/load utilities
- Plotting helpers
- Reproducibility (seed everything)

**How to use:** Run all cells once. All other notebooks import from this one via `%run`.

---
**⚠ NEVER change values mid-project.** All decisions here are fixed after preprocessing begins.
If you need to experiment, create a separate config copy and document the change.

## Cell 1 — Install & Imports

In [1]:
# ─── Install (run once on fresh Colab session) ────────────────────────────────

!pip install -q scipy scikit-learn numpy pandas matplotlib seaborn torch

# ─── Standard library ─────────────────────────────────────────────────────────
import os
import json
import random
import warnings
import itertools
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

# ─── Numerical / ML ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy import interpolate
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

# ─── Deep learning ────────────────────────────────────────────────────────────
import torch
import torch.nn as nn

# ─── Plotting ─────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
print('✅ Imports OK')
print(f'   PyTorch  : {torch.__version__}')
print(f'   NumPy    : {np.__version__}')
print(f'   CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU      : {torch.cuda.get_device_name(0)}')

✅ Imports OK
   PyTorch  : 2.10.0+cpu
   NumPy    : 2.0.2
   CUDA     : False


## Cell 2 — Reproducibility

In [2]:
# ─── Master seed ──────────────────────────────────────────────────────────────
# Used for ALL random operations. Change only if doing a sensitivity analysis run.
MASTER_SEED = 42

# Seeds to use for multi-run statistical significance (3–5 runs per config)
EVAL_SEEDS = [42, 7, 13, 21, 99]

def seed_everything(seed: int = MASTER_SEED) -> None:
    """Seed all RNGs for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(MASTER_SEED)
print(f'✅ Seeded everything with MASTER_SEED={MASTER_SEED}')

✅ Seeded everything with MASTER_SEED=42


## Cell 3 — Global Hyperparameters
These are the values decided during the full project design review.  
**Do not change without updating the thesis plan document.**

In [3]:
# ─── Preprocessing ────────────────────────────────────────────────────────────
TARGET_FREQ    = 50          # Hz — all sensors resampled to this rate
WINDOW_SIZE    = 250         # samples = 5 seconds × 50 Hz
WINDOW_STRIDE  = 125         # samples = 50% overlap

# ─── Few-shot episode settings ────────────────────────────────────────────────
# N_WAY: number of classes per episode
# Set to None to use min(5, num_classes_in_dataset) automatically
# Override per-dataset if needed in DATASET_CONFIGS below
N_WAY_DEFAULT  = 5
K_SHOTS        = [1, 5, 10]  # support set sizes to evaluate
N_QUERY        = 15          # query samples per class per episode
N_EPISODES_TRAIN = 300       # episodes per training epoch
N_EPISODES_EVAL  = 200       # episodes for evaluation

# ─── Training ─────────────────────────────────────────────────────────────────
BACKBONE_EPOCHS       = 50   # for pre-training backbone
FSL_EPOCHS            = 50   # for FSL meta-training
BATCH_SIZE            = 32
LEARNING_RATE         = 3e-4
EMBEDDING_DIM         = 128  # joint embedding size (output of all backbones)

# ─── MAML specific ────────────────────────────────────────────────────────────
MAML_INNER_LR         = 0.005
MAML_INNER_STEPS      = 3    # gradient steps in inner loop
MAML_OUTER_LR         = 5e-4

# ─── LEE specific ─────────────────────────────────────────────────────────────
LEE_LAMBDA            = 0.5  # weight of regularization loss vs task loss
LEE_FINETUNE_EPOCHS   = 10   # fine-tuning steps during personalization
LEE_FINETUNE_LR       = 1e-4

# ─── Data splits (by user) ────────────────────────────────────────────────────
SPLIT_TRAIN   = 0.75
SPLIT_VAL     = 0.125
SPLIT_TEST    = 0.125   # must sum to 1.0

# ─── Device ───────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('✅ Global hyperparameters set')
print(f'   Target frequency : {TARGET_FREQ} Hz')
print(f'   Window size      : {WINDOW_SIZE} samples ({WINDOW_SIZE/TARGET_FREQ:.1f} sec)')
print(f'   Window stride    : {WINDOW_STRIDE} samples ({WINDOW_STRIDE/TARGET_FREQ:.1f} sec)')
print(f'   K shots          : {K_SHOTS}')
print(f'   N way (default)  : {N_WAY_DEFAULT}')
print(f'   Embedding dim    : {EMBEDDING_DIM}')
print(f'   Device           : {DEVICE}')

✅ Global hyperparameters set
   Target frequency : 50 Hz
   Window size      : 250 samples (5.0 sec)
   Window stride    : 125 samples (2.5 sec)
   K shots          : [1, 5, 10]
   N way (default)  : 5
   Embedding dim    : 128
   Device           : cpu


## Cell 4 — Paths
All paths are defined here. If you move data, only change this cell.

In [4]:
# ─── Google Drive mount ───────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

# ─── Robust Base Path Detection ──────────────────────────────────────────────
import os

# Possible Google Drive root locations in Colab
possible_roots = [
    '/content/drive/My Drive',
    '/content/drive/MyDrive',
    '/content/gdrive/My Drive',
    '/content/gdrive/MyDrive'
]

colab_drive_root = None
for r in possible_roots:
    if os.path.exists(r):
        colab_drive_root = r
        break

if colab_drive_root is None:
    # Default fallback if nothing exists yet (e.g. before mounting)
    colab_drive_root = '/content/drive/My Drive'

# Detect if the project folder is nested or at the root of Google Drive
possible_bases = [
    colab_drive_root,
    os.path.join(colab_drive_root, 'Thesis_progress')
]

base_dir = colab_drive_root
for b in possible_bases:
    # If the results or processed data folder is found inside this base, use it!
    if os.path.exists(os.path.join(b, 'fsl_har_results')) or os.path.exists(os.path.join(b, 'fsl_har_processed')):
        base_dir = b
        break

print(f"✨ Smart Path Resolver active:")
print(f"   Drive Root: {colab_drive_root}")
print(f"   Project Base: {base_dir}")

# For RAW_PATHS, check if they are located under base_dir or colab_drive_root
def get_path(folder_name):
    path_under_project = os.path.join(base_dir, folder_name)
    if os.path.exists(path_under_project):
        return path_under_project
    return os.path.join(colab_drive_root, folder_name)

# ─── Dataset raw paths ────────────────────────────────────────────────────────
RAW_PATHS = {
    'wisdm': {
        'phone_accel' : os.path.join(get_path('wisdm-dataset'), 'raw/phone/accel/'),
        'phone_gyro'  : os.path.join(get_path('wisdm-dataset'), 'raw/phone/gyro/'),
        'watch_accel' : os.path.join(get_path('wisdm-dataset'), 'raw/watch/accel/'),
        'watch_gyro'  : os.path.join(get_path('wisdm-dataset'), 'raw/watch/gyro/'),
        'activity_key': os.path.join(get_path('wisdm-dataset'), 'activity_key.txt'),
    },
    'cogage_atomic': {
        'root': os.path.join(get_path('Annonmazed_CogActivity'), 'Sample/'),
    },
    'cogage_composite': {
        'root': get_path('sensory_data/'),
    },
    'humcare': {
        'root': get_path('DS_AF/DS_AF'),
    },
}

# ─── Processed data output paths ──────────────────────────────────────────────
PROCESSED_BASE = os.path.join(base_dir, 'fsl_har_processed')

PROCESSED_PATHS = {
    'wisdm'            : os.path.join(PROCESSED_BASE, 'wisdm'),
    'cogage_atomic'    : os.path.join(PROCESSED_BASE, 'cogage_atomic'),
    'cogage_composite' : os.path.join(PROCESSED_BASE, 'cogage_composite'),
    'humcare'          : os.path.join(PROCESSED_BASE, 'humcare'),
}

# ─── Model checkpoint paths ───────────────────────────────────────────────────
CHECKPOINT_BASE = os.path.join(base_dir, 'fsl_har_checkpoints')

# ─── Results paths ────────────────────────────────────────────────────────────
RESULTS_BASE = os.path.join(base_dir, 'fsl_har_results')

# ─── Create all output directories ───────────────────────────────────────────
for path in list(PROCESSED_PATHS.values()) + [CHECKPOINT_BASE, RESULTS_BASE]:
    os.makedirs(path, exist_ok=True)

print('✅ Paths configured')
print(f'   Processed data  → {PROCESSED_BASE}')
print(f'   Checkpoints     → {CHECKPOINT_BASE}')
print(f'   Results         → {RESULTS_BASE}')

Mounted at /content/drive
✅ Paths configured
   Processed data  → /content/drive/MyDrive/fsl_har_processed
   Checkpoints     → /content/drive/MyDrive/fsl_har_checkpoints
   Results         → /content/drive/MyDrive/fsl_har_results


## Cell 5 — Dataset Configs
Sensor streams, sampling rates, and activity labels for each dataset.

In [5]:
DATASET_CONFIGS = {

    # ── WISDM ─────────────────────────────────────────────────────────────────
    'wisdm': {
        'n_subjects'   : 51,
        'native_freq'  : 20,            # Hz — same for all 4 streams
        'n_way'        : 5,             # 5-way episodes
        'streams': {
            'phone_accel': {'freq': 20, 'axes': ['x','y','z']},
            'phone_gyro' : {'freq': 20, 'axes': ['x','y','z']},
            'watch_accel': {'freq': 20, 'axes': ['x','y','z']},
            'watch_gyro' : {'freq': 20, 'axes': ['x','y','z']},
        },
        'activity_labels': {
            'A':'walking',   'B':'jogging',   'C':'stairs',
            'D':'sitting',   'E':'standing',  'F':'typing',
            'G':'teeth',     'H':'soup',      'I':'chips',
            'J':'pasta',     'K':'drinking',  'L':'sandwich',
            'M':'kicking',   'O':'catch',     'P':'dribbling',
            'Q':'writing',   'R':'clapping',  'S':'folding',
        },
        # Map letter labels → integer indices
        'label_to_int': {
            'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6,'H':7,'I':8,
            'J':9,'K':10,'L':11,'M':12,'O':13,'P':14,'Q':15,'R':16,'S':17,
        },
        'n_classes'    : 18,
        'n_streams'    : 4,
    },

    # ── CogAge Atomic ─────────────────────────────────────────────────────────
    'cogage_atomic': {
        'n_subjects'   : 8,
        'n_way'        : 5,
        'streams': {
            # JINS glasses
            'jins_accel'         : {'freq': 20,  'axes': ['x','y','z']},
            'jins_gyro'          : {'freq': 20,  'axes': ['x','y','z']},
            # Phone
            'phone_accel'        : {'freq': 200, 'axes': ['x','y','z']},
            'phone_gyro'         : {'freq': 200, 'axes': ['x','y','z']},
            'phone_gravity'      : {'freq': 200, 'axes': ['x','y','z']},
            'phone_linear_accel' : {'freq': 200, 'axes': ['x','y','z']},
            'phone_magnetometer' : {'freq': 50,  'axes': ['x','y','z']},
            # Watch (Microsoft Band)
            'ms_accel'           : {'freq': 67,  'axes': ['x','y','z']},
            'ms_gyro'            : {'freq': 67,  'axes': ['x','y','z']},
            # Removed: JinsBlinkStrength, JinsEyeMovement, JinsBlinkSpeed
        },
        # Session 1 = train, Session 2 = test (from folder name)
        'session_split': {1: 'train', 2: 'test'},
        'activity_labels': {
            0:'Bending',           1:'Lying',              2:'Sitting',
            3:'Squatting',         4:'Standing',           5:'Walking',
            6:'Bring',             7:'CleanFloor',         8:'CleanSurface',
            9:'CloseBigBox',       10:'CloseDoor',         11:'CloseDrawer',
            12:'CloseLidByRotate', 13:'CloseOtherLid',     14:'CloseSmallBox',
            15:'CloseTapWater',    16:'Drink',             17:'DryOffHand',
            18:'DryOffHandByShake',19:'EatSmall',          20:'Gargle',
            21:'GettingUp',        22:'Hang',              23:'LyingDown',
            24:'OpenBag',          25:'OpenBigBox',        26:'OpenDoor',
            27:'OpenDrawer',       28:'OpenLidByRotate',   29:'OpenOtherLid',
            30:'OpenSmallBox',     31:'OpenTapWater',      32:'PlugIn',
            33:'PressByGrasp',     34:'PressFromTop',      35:'PressSwitch',
            36:'PutFromBottle',    37:'PutFromTapWater',   38:'PutHighPosition',
            39:'PutOnFloor',       40:'Read',              41:'Rotate',
            42:'RubHands',         43:'ScoopPut',          44:'SittingDown',
            45:'SquattingDown',    46:'StandingUp',        47:'StandUpFromSquatting',
            48:'TakeFromFloor',    49:'TakeFromHighPosition',50:'TakeOffJacket',
            51:'TakeOut',          52:'TalkByTelephone',   53:'ThrowOut',
            54:'ThrowOutWater',    55:'TouchSmartPhoneScreen',56:'Type',
            57:'Unhang',           58:'Unplug',            59:'WearJacket',
            60:'Write',
        },
        'n_classes'    : 61,
        'n_streams'    : 9,
        'activity_duration_sec': 5,   # fixed 5-second activities
        'n_repetitions': 10,          # 0–9
    },

    # ── CogAge Composite ──────────────────────────────────────────────────────
    'cogage_composite': {
        'n_subjects'   : 6,           # after removing subjects 7 and 8
        'removed_subjects': [7, 8],   # 7=no watch data, 8=idle states
        'n_way'        : 5,           # only 7 classes — use min(5,7)
        'streams': {
            # Phone (7 streams)
            'phone_accel'        : {'freq': None, 'axes': ['x','y','z']},  # freq TBD from files
            'phone_barometer'    : {'freq': None, 'axes': ['x','y','z']},
            'phone_gravity'      : {'freq': None, 'axes': ['x','y','z']},
            'phone_gyro'         : {'freq': None, 'axes': ['x','y','z']},
            'phone_linear_accel' : {'freq': None, 'axes': ['x','y','z']},
            'phone_magnetometer' : {'freq': None, 'axes': ['x','y','z']},
            'phone_orientation'  : {'freq': None, 'axes': ['x','y','z']},
            # Watch (2 streams)
            'watch_accel'        : {'freq': None, 'axes': ['x','y','z']},
            'watch_gyro'         : {'freq': None, 'axes': ['x','y','z']},
            # Glasses (1 stream) — JINS: keep X,Y,Z only
            'jins_accel'         : {'freq': None, 'axes': ['x','y','z']},
        },
        'activity_labels': {
            0:'brushing_teeth',
            1:'cleaning_room',
            2:'handling_medications',
            3:'preparing_food',
            4:'styling_hair',
            5:'using_telephone',
            6:'washing_hands',
        },
        'n_classes'    : 7,
        'n_streams'    : 10,
        # Subjects with irregular structure requiring manual fix
        'special_subjects': {
            2: 'chronological_split',   # has Lefthand/Both/Mixed → keep Lefthand, split 70/30
            4: 'rename_left',           # has left/both/right → delete both+right, rename left→lefthand
        },
        # Episode construction note:
        # Each folder = one complete composite activity recording → apply sliding window
        # Do NOT mix windows from different activities in one episode
    },

    # ── Humcare AF ────────────────────────────────────────────────────────────
    'humcare': {
        'n_subjects'   : 87,
        'n_way'        : 5,
        'streams': {
            'phone_accel'       : {'freq': 400, 'axes': ['x','y','z']},
            'phone_gyro'        : {'freq': 400, 'axes': ['x','y','z']},
            'phone_magnetometer': {'freq': 100, 'axes': ['x','y','z']},
            'watch_accel'       : {'freq': 100, 'axes': ['x','y','z']},
            'watch_gyro'        : {'freq': 100, 'axes': ['x','y','z']},
            'watch_magnetometer': {'freq': 100, 'axes': ['x','y','z']},
            'glass_accel'       : {'freq': 5,   'axes': ['x','y','z']},
            'glass_gyro'        : {'freq': 5,   'axes': ['x','y','z']},
            'glass_magnetometer': {'freq': 5,   'axes': ['x','y','z']},
        },
        'activity_labels': {
            0:'walking',                    1:'slow_walk',
            2:'fast_walk',                  3:'jogging',
            4:'up_stairs',                  5:'down_stairs',
            6:'sitting',                    7:'standing',
            8:'laying',                     9:'bending',
            10:'standing_up_from_sitting',  11:'standing_up_from_lying',
            12:'lying_down_from_sitting',   13:'sitting_down_from_standing',
            14:'squatting',                 15:'typing',
            16:'clean_the_table',           17:'reading',
            18:'talk_using_phone',          19:'drink_water',
            20:'open_door',                 21:'close_door',
            22:'pick_from_floor',           23:'put_on_floor',
            24:'open_big_box',              25:'close_lid_by_rotation',
            26:'open_bag',                  27:'eat_small_things',
            28:'plug_in',                   29:'throw_out',
            30:'fall_forward',              31:'fall_right',
            32:'fall_backward',             33:'fall_left',
            34:'fall_forward_sitting_down', 35:'fall_backward_sitting_down',
            36:'fall_forward_standing_up',  37:'fall_backward_standing_up',
        },
        'n_classes'    : 38,
        'n_streams'    : 9,
        # Not all subjects performed all activities — handled by subject_activity_map
        'inconsistent_coverage': True,
        # Minimum valid window = sensor_freq × 4 seconds
        'min_window_sec': 4,
    },
}

# Convenience: list of all dataset names
ALL_DATASETS   = list(DATASET_CONFIGS.keys())
ALL_BACKBONES  = ['cnn', 'lstm', 'transformer']
ALL_FSL_METHODS = ['protonet', 'lee', 'maml', 'supcon']

print('✅ Dataset configs loaded')
for name, cfg in DATASET_CONFIGS.items():
    print(f'   {name:<22} → {cfg["n_subjects"]:>3} subjects | '
          f'{cfg["n_classes"]:>3} classes | '
          f'{cfg["n_streams"]:>2} streams | '
          f'n_way={cfg["n_way"]}')

✅ Dataset configs loaded
   wisdm                  →  51 subjects |  18 classes |  4 streams | n_way=5
   cogage_atomic          →   8 subjects |  61 classes |  9 streams | n_way=5
   cogage_composite       →   6 subjects |   7 classes | 10 streams | n_way=5
   humcare                →  87 subjects |  38 classes |  9 streams | n_way=5


## Cell 6 — User-Level Train/Val/Test Split
Splits are always by subject ID. Never by sample.

In [6]:
def split_subjects(
    subject_ids: List,
    train_ratio: float = SPLIT_TRAIN,
    val_ratio:   float = SPLIT_VAL,
    seed:        int   = MASTER_SEED
) -> Tuple[List, List, List]:
    """
    Split a list of subject IDs into train / val / test at the subject level.
    No subject appears in more than one split.

    Args:
        subject_ids : full list of subject IDs for this dataset
        train_ratio : fraction of subjects for training
        val_ratio   : fraction of subjects for validation
        seed        : random seed for reproducibility

    Returns:
        (train_subjects, val_subjects, test_subjects)
    """
    assert abs(train_ratio + val_ratio + (1 - train_ratio - val_ratio) - 1.0) < 1e-6

    rng = np.random.default_rng(seed)
    ids = list(subject_ids)
    rng.shuffle(ids)

    n_total = len(ids)
    n_train = max(1, int(np.floor(n_total * train_ratio)))
    n_val   = max(1, int(np.floor(n_total * val_ratio)))
    # test gets the remainder
    n_test  = n_total - n_train - n_val

    if n_test < 1:
        raise ValueError(
            f'Not enough subjects ({n_total}) for the requested split ratios. '
            f'Got train={n_train}, val={n_val}, test={n_test}.'
        )

    train_subjects = ids[:n_train]
    val_subjects   = ids[n_train:n_train + n_val]
    test_subjects  = ids[n_train + n_val:]

    # Sanity check — no overlap
    assert len(set(train_subjects) & set(val_subjects))  == 0, 'Overlap: train/val'
    assert len(set(train_subjects) & set(test_subjects)) == 0, 'Overlap: train/test'
    assert len(set(val_subjects)   & set(test_subjects)) == 0, 'Overlap: val/test'

    return train_subjects, val_subjects, test_subjects


def get_split_mask(
    subject_ids_array: np.ndarray,
    target_subjects:   List
) -> np.ndarray:
    """
    Return a boolean mask for windows belonging to target subjects.

    Args:
        subject_ids_array : (N,) array of subject IDs per window
        target_subjects   : list of subject IDs to select

    Returns:
        Boolean mask of shape (N,)
    """
    return np.isin(subject_ids_array, target_subjects)


# ─── Quick demo ───────────────────────────────────────────────────────────────
demo_subjects = list(range(1, 52))  # WISDM: subjects 1–51
tr, va, te = split_subjects(demo_subjects)
print('✅ Subject split utility OK')
print(f'   WISDM demo  → train={len(tr)} | val={len(va)} | test={len(te)} subjects')
print(f'   Train : {sorted(tr)[:5]}...')
print(f'   Val   : {sorted(va)}')
print(f'   Test  : {sorted(te)}')

✅ Subject split utility OK
   WISDM demo  → train=38 | val=6 | test=7 subjects
   Train : [4, 5, 6, 7, 8]...
   Val   : [1, 13, 15, 20, 23, 36]
   Test  : [2, 3, 9, 14, 34, 37, 50]


## Cell 7 — Resampling Utility
Resamples a single sensor stream from its native frequency to TARGET_FREQ.

In [7]:
def resample_stream(
    data:       np.ndarray,
    src_freq:   float,
    tgt_freq:   float = TARGET_FREQ,
    kind:       str   = 'linear'
) -> np.ndarray:
    """
    Resample a 2D sensor array from src_freq to tgt_freq using interpolation.

    Args:
        data     : (T_src, C) array — T_src samples, C axes
        src_freq : original sampling frequency in Hz
        tgt_freq : target sampling frequency in Hz (default TARGET_FREQ=50)
        kind     : interpolation kind ('linear', 'cubic') — use 'linear' always
                   for low-frequency signals (glasses at 5Hz) to avoid overfitting

    Returns:
        (T_tgt, C) array at tgt_freq

    Notes:
        - Upsampling (5Hz → 50Hz for glasses): linear interpolation only.
          This produces smooth but uninformative intermediate samples.
          The model is expected to learn to downweight these channels.
        - Downsampling (400Hz → 50Hz for phone): interpolate then subsample.
    """
    if src_freq == tgt_freq:
        return data.copy()

    T_src, C = data.shape

    # Time axes
    duration   = T_src / src_freq          # seconds
    t_src      = np.linspace(0, duration, T_src, endpoint=False)
    T_tgt      = max(2, int(round(duration * tgt_freq)))
    t_tgt      = np.linspace(0, duration, T_tgt, endpoint=False)

    resampled = np.zeros((T_tgt, C), dtype=np.float32)
    for c in range(C):
        f = interpolate.interp1d(
            t_src, data[:, c],
            kind=kind,
            bounds_error=False,
            fill_value=(data[0, c], data[-1, c])   # edge padding
        )
        resampled[:, c] = f(t_tgt)

    return resampled


# ─── Quick test ───────────────────────────────────────────────────────────────
dummy_400hz = np.random.randn(2000, 3).astype(np.float32)  # 5 sec at 400Hz
dummy_5hz   = np.random.randn(25, 3).astype(np.float32)    # 5 sec at 5Hz

out_400 = resample_stream(dummy_400hz, src_freq=400)  # 400→50
out_5   = resample_stream(dummy_5hz,   src_freq=5)    # 5→50

print('✅ Resample utility OK')
print(f'   400Hz → 50Hz : (2000,3) → {out_400.shape}  (expected ~(250,3))')
print(f'     5Hz → 50Hz :   (25,3) → {out_5.shape}   (expected ~(250,3))')

✅ Resample utility OK
   400Hz → 50Hz : (2000,3) → (250, 3)  (expected ~(250,3))
     5Hz → 50Hz :   (25,3) → (250, 3)   (expected ~(250,3))


## Cell 8 — Sliding Window Utility

In [8]:
def sliding_window(
    data:        np.ndarray,
    label:       int,
    subject_id,
    window_size: int   = WINDOW_SIZE,
    stride:      int   = WINDOW_STRIDE,
    min_length:  int   = None
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Apply sliding window to a single continuous sensor stream recording.

    Args:
        data        : (T, C) array — T timesteps, C axes (already at TARGET_FREQ)
        label       : integer class label for this recording
        subject_id  : subject identifier
        window_size : number of samples per window (default WINDOW_SIZE=250)
        stride      : step between windows (default WINDOW_STRIDE=125, 50% overlap)
        min_length  : minimum required length to process; skip if shorter
                      (used for Humcare: sensor_freq × 4)

    Returns:
        windows     : (N_windows, window_size, C)
        labels      : (N_windows,)  — all equal to `label`
        subjects    : (N_windows,)  — all equal to `subject_id`

    Notes:
        - If recording is shorter than window_size → returns empty arrays
        - Last incomplete window is dropped (not zero-padded)
    """
    T, C = data.shape

    # Check minimum length
    if min_length is not None and T < min_length:
        return (np.empty((0, window_size, C), dtype=np.float32),
                np.empty((0,), dtype=np.int64),
                np.empty((0,), dtype=object))

    if T < window_size:
        return (np.empty((0, window_size, C), dtype=np.float32),
                np.empty((0,), dtype=np.int64),
                np.empty((0,), dtype=object))

    starts  = range(0, T - window_size + 1, stride)
    windows = np.stack([data[s:s + window_size] for s in starts], axis=0)
    labels  = np.full(len(windows), label,      dtype=np.int64)
    subjects= np.full(len(windows), subject_id, dtype=object)

    return windows.astype(np.float32), labels, subjects


# ─── Quick test ───────────────────────────────────────────────────────────────
dummy_recording = np.random.randn(600, 3).astype(np.float32)  # ~12 sec at 50Hz
wins, labs, subs = sliding_window(dummy_recording, label=0, subject_id='S1')
print('✅ Sliding window utility OK')
print(f'   Input  : (600, 3)')
print(f'   Output : windows={wins.shape} | labels={labs.shape} | subjects={subs.shape}')
print(f'   Expected ~3 windows: {(600 - 250) // 125 + 1} windows')

✅ Sliding window utility OK
   Input  : (600, 3)
   Output : windows=(3, 250, 3) | labels=(3,) | subjects=(3,)
   Expected ~3 windows: 3 windows


## Cell 9 — Normalisation Utility
Per-stream z-score normalisation fitted on training windows only.

In [9]:
class StreamNormalizer:
    """
    Z-score normalizer for a single sensor stream.
    Fit on training windows, transform train/val/test.

    Usage:
        norm = StreamNormalizer()
        X_train_norm = norm.fit_transform(X_train)  # (N, T, C)
        X_test_norm  = norm.transform(X_test)
    """

    def __init__(self):
        self.mean_ = None   # (C,)
        self.std_  = None   # (C,)

    def fit(self, X: np.ndarray) -> 'StreamNormalizer':
        """
        Fit on training data.
        X : (N, T, C) — N windows, T timesteps, C axes
        Statistics computed over N and T dimensions, per channel C.
        """
        # Reshape to (N*T, C) for statistics
        flat = X.reshape(-1, X.shape[-1])
        self.mean_ = flat.mean(axis=0)
        self.std_  = flat.std(axis=0)
        # Avoid division by zero for static channels
        self.std_  = np.where(self.std_ < 1e-8, 1.0, self.std_)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        """Apply normalisation. Returns same shape as input."""
        if self.mean_ is None:
            raise RuntimeError('StreamNormalizer must be fit before transform.')
        return ((X - self.mean_) / self.std_).astype(np.float32)

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        return self.fit(X).transform(X)

    def save(self, path: str) -> None:
        np.save(path + '_mean.npy', self.mean_)
        np.save(path + '_std.npy',  self.std_)

    def load(self, path: str) -> 'StreamNormalizer':
        self.mean_ = np.load(path + '_mean.npy')
        self.std_  = np.load(path + '_std.npy')
        return self


# ─── Quick test ───────────────────────────────────────────────────────────────
dummy_X = np.random.randn(100, 250, 3).astype(np.float32)
norm    = StreamNormalizer()
X_norm  = norm.fit_transform(dummy_X)
print('✅ StreamNormalizer OK')
print(f'   Input mean  : {dummy_X.mean():.3f} → normalised mean  : {X_norm.mean():.4f}')
print(f'   Input std   : {dummy_X.std():.3f}  → normalised std   : {X_norm.std():.4f}')

✅ StreamNormalizer OK
   Input mean  : 0.002 → normalised mean  : 0.0000
   Input std   : 1.001  → normalised std   : 1.0000


## Cell 10 — Save & Load Utilities
Standardised functions for saving preprocessed arrays and loading them back.

In [10]:
def save_processed_dataset(
    dataset_name : str,
    stream_arrays: Dict[str, np.ndarray],
    y            : np.ndarray,
    subject_ids  : np.ndarray,
    stream_names : List[str],
    subject_activity_map: Optional[Dict] = None,
    split_info   : Optional[Dict] = None,
) -> None:
    """
    Save all outputs of a preprocessing notebook.

    Saves to PROCESSED_PATHS[dataset_name]/
        {stream_name}.npy       — (N, T, 3) per stream
        y.npy                   — (N,) labels
        subject_ids.npy         — (N,) subject IDs
        stream_names.json       — ordered list of stream names
        subject_activity_map.json  — Humcare only
        split_info.json         — train/val/test subject lists
    """
    out_dir = PROCESSED_PATHS[dataset_name]
    os.makedirs(out_dir, exist_ok=True)

    # Save each stream
    for name, arr in stream_arrays.items():
        path = os.path.join(out_dir, f'{name}.npy')
        np.save(path, arr)
        print(f'   Saved stream {name:<25} → {arr.shape}  →  {path}')

    # Save labels and subject IDs
    np.save(os.path.join(out_dir, 'y.npy'),           y)
    np.save(os.path.join(out_dir, 'subject_ids.npy'), subject_ids)
    print(f'   Saved y.npy          → {y.shape}')
    print(f'   Saved subject_ids    → {subject_ids.shape}')

    # Save stream names list
    with open(os.path.join(out_dir, 'stream_names.json'), 'w') as f:
        json.dump(stream_names, f, indent=2)

    # Save subject-activity map (Humcare)
    if subject_activity_map is not None:
        with open(os.path.join(out_dir, 'subject_activity_map.json'), 'w') as f:
            json.dump({str(k): v for k, v in subject_activity_map.items()}, f, indent=2)
        print(f'   Saved subject_activity_map.json')

    # Save split info
    if split_info is not None:
        with open(os.path.join(out_dir, 'split_info.json'), 'w') as f:
            json.dump({k: [str(s) for s in v] for k, v in split_info.items()}, f, indent=2)
        print(f'   Saved split_info.json')

    print(f'\n✅ Dataset "{dataset_name}" saved to {out_dir}')


def load_processed_dataset(
    dataset_name: str,
    streams_to_load: Optional[List[str]] = None
) -> Dict:
    """
    Load a preprocessed dataset from disk.

    Args:
        dataset_name    : one of 'wisdm', 'cogage_atomic', 'cogage_composite', 'humcare'
        streams_to_load : list of stream names to load (None = load all)

    Returns dict with keys:
        'streams'               : {stream_name: (N,T,3) array}
        'y'                     : (N,) labels
        'subject_ids'           : (N,) subject IDs
        'stream_names'          : ordered list of stream names
        'subject_activity_map'  : dict (Humcare only, else None)
        'split_info'            : dict with train/val/test subject lists
        'config'                : DATASET_CONFIGS[dataset_name]
    """
    out_dir = PROCESSED_PATHS[dataset_name]

    # Load stream names
    with open(os.path.join(out_dir, 'stream_names.json')) as f:
        stream_names = json.load(f)

    # Load streams
    to_load = streams_to_load if streams_to_load else stream_names
    streams = {}
    for name in to_load:
        path = os.path.join(out_dir, f'{name}.npy')
        streams[name] = np.load(path)

    # Load labels and subjects
    y           = np.load(os.path.join(out_dir, 'y.npy'))
    subject_ids = np.load(os.path.join(out_dir, 'subject_ids.npy'), allow_pickle=True)

    # Load optional files
    sam_path = os.path.join(out_dir, 'subject_activity_map.json')
    subject_activity_map = None
    if os.path.exists(sam_path):
        with open(sam_path) as f:
            subject_activity_map = json.load(f)

    split_path = os.path.join(out_dir, 'split_info.json')
    split_info = None
    if os.path.exists(split_path):
        with open(split_path) as f:
            split_info = json.load(f)

    print(f'✅ Loaded "{dataset_name}" — {len(y)} windows | {len(streams)} streams')
    return {
        'streams'              : streams,
        'y'                    : y,
        'subject_ids'          : subject_ids,
        'stream_names'         : stream_names,
        'subject_activity_map' : subject_activity_map,
        'split_info'           : split_info,
        'config'               : DATASET_CONFIGS[dataset_name],
    }


def save_checkpoint(model: nn.Module, path: str, extra: Dict = None) -> None:
    """Save model weights + optional metadata dict."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    payload = {'state_dict': model.state_dict()}
    if extra:
        payload['meta'] = extra
    torch.save(payload, path)


def load_checkpoint(model: nn.Module, path: str) -> Dict:
    """Load model weights from checkpoint. Returns metadata if present."""
    payload = torch.load(path, map_location=DEVICE)
    model.load_state_dict(payload['state_dict'])
    return payload.get('meta', {})


GEN_COLUMNS = [
    'dataset',
    'backbone',
    'method',
    'eval_type',
    'k_shot',
    'seed',
    'acc_mean',
    'acc_std',
    'ci95',
    'macro_f1',
    'n_episodes'
]

PERS_COLUMNS = [
    'dataset',
    'method',
    'backbone',
    'k_shot',
    'subject',
    'acc_before',
    'acc_after',
    'gain',
    'ci95'
    'n_trials'
]

def append_gen_results(results_dict: Dict, path: str) -> None:
    import pandas as pd, os
    os.makedirs(os.path.dirname(path), exist_ok=True)

    row_fixed = {col: results_dict.get(col, None) for col in GEN_COLUMNS}
    row = pd.DataFrame([row_fixed], columns=GEN_COLUMNS)

    if os.path.exists(path):
        df_old = pd.read_csv(path)

        if list(df_old.columns) != GEN_COLUMNS:
            raise ValueError(f"Schema mismatch in {path}")

        df = pd.concat([df_old, row], ignore_index=True)
    else:
        df = row

    df.to_csv(path, index=False)
def append_pers_results(results_dict: Dict, path: str) -> None:
    import pandas as pd, os
    os.makedirs(os.path.dirname(path), exist_ok=True)

    row_fixed = {col: results_dict.get(col, None) for col in PERS_COLUMNS}
    row = pd.DataFrame([row_fixed], columns=PERS_COLUMNS)

    if os.path.exists(path):
        df_old = pd.read_csv(path)

        if list(df_old.columns) != PERS_COLUMNS:
            raise ValueError(f"Schema mismatch in {path}")

        df = pd.concat([df_old, row], ignore_index=True)
    else:
        df = row

    df.to_csv(path, index=False)


print('✅ Save/load utilities OK')

✅ Save/load utilities OK


## Cell 11 — Episode Sampler
N-way K-shot episode construction for all FSL methods.

In [11]:
class EpisodeSampler:
    """
    N-way K-shot episode sampler.

    Handles:
    - Standard datasets (all subjects have all activities)
    - Humcare (inconsistent coverage via subject_activity_map)

    An episode contains:
        support set : N classes × K shots    = N*K windows
        query set   : N classes × N_QUERY    = N*N_QUERY windows

    All windows in an episode come from the SAME split (train/val/test).
    Labels in each episode are remapped 0..N-1 (relative labels).
    """

    def __init__(
        self,
        stream_arrays        : Dict[str, np.ndarray],
        y                    : np.ndarray,
        subject_ids          : np.ndarray,
        subject_split        : List,            # list of subject IDs for this split
        n_way                : int   = N_WAY_DEFAULT,
        n_query              : int   = N_QUERY,
        subject_activity_map : Optional[Dict] = None,
        seed                 : int   = MASTER_SEED,
    ):
        """
        Args:
            stream_arrays        : {stream_name: (N, T, 3)} — all windows
            y                    : (N,) global labels
            subject_ids          : (N,) subject IDs
            subject_split        : subjects belonging to this split (train/val/test)
            n_way                : classes per episode
            n_query              : query samples per class per episode
            subject_activity_map : {subject_id: [activity_ids]} for Humcare
            seed                 : for episode reproducibility
        """
        self.stream_arrays = stream_arrays
        self.n_way   = n_way
        self.n_query = n_query
        self.rng     = np.random.default_rng(seed)

        # Filter to split subjects only
        split_mask   = get_split_mask(subject_ids, subject_split)
        self.y       = y[split_mask]
        self.sub_ids = subject_ids[split_mask]
        self.split_stream_arrays = {
            k: v[split_mask] for k, v in stream_arrays.items()
        }

        # Build class → indices map
        self.class_to_indices: Dict[int, np.ndarray] = defaultdict(list)
        for idx, label in enumerate(self.y):
            self.class_to_indices[int(label)].append(idx)
        self.class_to_indices = {
            k: np.array(v) for k, v in self.class_to_indices.items()
        }

        # If Humcare: identify which classes have enough subjects
        self.subject_activity_map = subject_activity_map
        self.available_classes = self._get_available_classes()

        if len(self.available_classes) < self.n_way:
            print(f'⚠ Only {len(self.available_classes)} classes available '
                  f'but n_way={self.n_way}. '
                  f'Reducing n_way to {len(self.available_classes)}.')
            self.n_way = len(self.available_classes)

    def _get_available_classes(self) -> List[int]:
        """
        Classes that have at least (K_shot_max + N_QUERY) samples in this split.
        K_shot_max = max(K_SHOTS) = 10.
        """
        min_required = max(K_SHOTS) + self.n_query
        return [
            cls for cls, idxs in self.class_to_indices.items()
            if len(idxs) >= min_required
        ]

    def sample_episode(
        self,
        k_shot: int
    ) -> Dict[str, torch.Tensor]:
        """
        Sample one N-way K-shot episode.

        Args:
            k_shot : number of support samples per class

        Returns dict:
            'support_streams' : {stream_name: (N_way*K, T, 3)} tensors
            'query_streams'   : {stream_name: (N_way*N_query, T, 3)} tensors
            'support_labels'  : (N_way*K,)       — relative labels 0..N_way-1
            'query_labels'    : (N_way*N_query,)  — relative labels 0..N_way-1
            'global_classes'  : list of N_way global class IDs selected
        """
        assert k_shot in K_SHOTS, f'k_shot must be one of {K_SHOTS}, got {k_shot}'

        # Sample N_way classes
        chosen_classes = self.rng.choice(
            self.available_classes, size=self.n_way, replace=False
        ).tolist()

        support_idxs = []
        query_idxs   = []
        support_labs = []
        query_labs   = []

        for rel_label, global_class in enumerate(chosen_classes):
            class_idxs = self.class_to_indices[global_class].copy()
            self.rng.shuffle(class_idxs)

            needed = k_shot + self.n_query
            if len(class_idxs) < needed:
                # Sample with replacement if not enough (rare edge case)
                class_idxs = self.rng.choice(class_idxs, size=needed, replace=True)

            support_idxs.extend(class_idxs[:k_shot].tolist())
            query_idxs.extend(class_idxs[k_shot:k_shot + self.n_query].tolist())
            support_labs.extend([rel_label] * k_shot)
            query_labs.extend([rel_label] * self.n_query)

        # Build stream tensors
        def gather_streams(indices):
            return {
                name: torch.tensor(
                    arr[indices], dtype=torch.float32
                )
                for name, arr in self.split_stream_arrays.items()
            }

        return {
            'support_streams' : gather_streams(support_idxs),
            'query_streams'   : gather_streams(query_idxs),
            'support_labels'  : torch.tensor(support_labs, dtype=torch.long),
            'query_labels'    : torch.tensor(query_labs,   dtype=torch.long),
            'global_classes'  : chosen_classes,
        }


print('✅ EpisodeSampler class defined')

✅ EpisodeSampler class defined


## Cell 12 — Metrics

In [12]:
def compute_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Standard accuracy."""
    return float(accuracy_score(y_true, y_pred))


def compute_macro_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Macro-averaged F1 score — handles class imbalance."""
    return float(f1_score(y_true, y_pred, average='macro', zero_division=0))


def compute_personalization_gain(
    acc_before: float,
    acc_after:  float
) -> float:
    """
    Personalization gain = accuracy after adaptation − accuracy before adaptation.
    Positive = adaptation helped. Negative = adaptation hurt (regression).
    """
    return acc_after - acc_before


def summarise_results(
    accs_before  : List[float],
    accs_after   : List[float],
    f1s_before   : List[float] = None,
    f1s_after    : List[float] = None,
) -> Dict:
    """
    Summarise multi-seed results into mean ± std.

    Returns dict with:
        acc_before_mean, acc_before_std
        acc_after_mean,  acc_after_std
        gain_mean,       gain_std
        f1_before_mean,  f1_after_mean  (if provided)
    """
    gains = [compute_personalization_gain(b, a)
             for b, a in zip(accs_before, accs_after)]
    out = {
        'acc_before_mean' : float(np.mean(accs_before)),
        'acc_before_std'  : float(np.std(accs_before)),
        'acc_after_mean'  : float(np.mean(accs_after)),
        'acc_after_std'   : float(np.std(accs_after)),
        'gain_mean'       : float(np.mean(gains)),
        'gain_std'        : float(np.std(gains)),
    }
    if f1s_before and f1s_after:
        out['f1_before_mean'] = float(np.mean(f1s_before))
        out['f1_after_mean']  = float(np.mean(f1s_after))
    return out


# ─── Quick test ───────────────────────────────────────────────────────────────
y_t = np.array([0,0,1,1,2,2])
y_p = np.array([0,1,1,1,2,0])
print('✅ Metrics OK')
print(f'   Accuracy : {compute_accuracy(y_t, y_p):.4f}  (expected {4/6:.4f})')
print(f'   Macro F1 : {compute_macro_f1(y_t, y_p):.4f}')
print(f'   Pers gain: {compute_personalization_gain(0.60, 0.75):.4f}  (expected 0.15)')

✅ Metrics OK
   Accuracy : 0.6667  (expected 0.6667)
   Macro F1 : 0.6556
   Pers gain: 0.1500  (expected 0.15)


## Cell 13 — Plotting Helpers
Consistent style across all result visualizations.

In [13]:
# ─── Consistent plot style ────────────────────────────────────────────────────
PLOT_STYLE = {
    'figure.figsize'     : (10, 6),
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
    'axes.grid'          : True,
    'grid.alpha'         : 0.3,
    'font.size'          : 12,
}
plt.rcParams.update(PLOT_STYLE)

METHOD_COLORS = {
    'protonet' : '#2196F3',   # blue
    'lee'      : '#FF5722',   # orange
    'maml'     : '#4CAF50',   # green
    'supcon'   : '#9C27B0',   # purple
}
BACKBONE_MARKERS = {
    'cnn'         : 'o',
    'lstm'        : 's',
    'transformer' : '^',
}


def plot_k_shot_curves(
    results: Dict,
    dataset: str,
    backbone: str,
    metric: str = 'acc_after_mean',
    save_path: str = None
) -> None:
    """
    Plot K-shot accuracy curves for all FSL methods.

    Args:
        results  : {method: {k_shot: summary_dict}}
        dataset  : dataset name (for title)
        backbone : backbone name (for title)
        metric   : which metric to plot
        save_path: if provided, save figure to this path
    """
    fig, ax = plt.subplots()
    for method, k_results in results.items():
        ks   = sorted(k_results.keys())
        vals = [k_results[k][metric] for k in ks]
        stds = [k_results[k].get(metric.replace('mean','std'), 0) for k in ks]
        ax.errorbar(
            ks, vals, yerr=stds,
            label=method.upper(),
            color=METHOD_COLORS.get(method, 'gray'),
            marker=BACKBONE_MARKERS.get(backbone, 'o'),
            linewidth=2, markersize=8, capsize=4
        )
    ax.set_xlabel('K (support shots per class)')
    ax.set_ylabel(metric.replace('_', ' ').title())
    ax.set_title(f'{dataset.upper()} | {backbone.upper()} | K-shot curves')
    ax.set_xticks(K_SHOTS)
    ax.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_personalization_gain_heatmap(
    gain_matrix: np.ndarray,
    row_labels:  List[str],
    col_labels:  List[str],
    title:       str,
    save_path:   str = None
) -> None:
    """
    Heatmap of personalization gains.
    Rows = methods, Cols = datasets (or backbones).
    Green = positive gain, Red = regression.
    """
    fig, ax = plt.subplots(figsize=(len(col_labels)*2+2, len(row_labels)*1.2+2))
    sns.heatmap(
        gain_matrix,
        annot=True, fmt='.3f',
        xticklabels=col_labels,
        yticklabels=row_labels,
        cmap='RdYlGn',
        center=0,
        linewidths=0.5,
        ax=ax
    )
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


print('✅ Plotting helpers OK')

✅ Plotting helpers OK


## Cell 14 — Subject-Activity Map Builder
For Humcare AF: builds the map of which subjects performed which activities.

In [14]:
def build_subject_activity_map(
    subject_ids : np.ndarray,
    y           : np.ndarray,
) -> Dict:
    """
    Build a map of {subject_id: [list of activity IDs performed]}.
    Used by EpisodeSampler for Humcare where coverage is inconsistent.

    Args:
        subject_ids : (N,) array of subject IDs per window
        y           : (N,) array of activity labels per window

    Returns:
        dict mapping each subject to sorted list of unique activities they performed
    """
    sam = defaultdict(set)
    for subj, act in zip(subject_ids, y):
        sam[str(subj)].add(int(act))
    return {k: sorted(list(v)) for k, v in sam.items()}


def get_subjects_for_activity(
    activity_id          : int,
    subject_activity_map : Dict,
    split_subjects       : List
) -> List:
    """
    Return subjects in split_subjects who performed activity_id.
    Used during episode construction for Humcare.
    """
    return [
        s for s in split_subjects
        if str(s) in subject_activity_map
        and activity_id in subject_activity_map[str(s)]
    ]


# ─── Demo ─────────────────────────────────────────────────────────────────────
demo_subs = np.array(['s1','s1','s1','s2','s2','s3','s3','s3'])
demo_y    = np.array([0, 0, 1, 1, 2, 0, 2, 3])
demo_map  = build_subject_activity_map(demo_subs, demo_y)
print('✅ Subject-activity map builder OK')
print(f'   Demo map: {demo_map}')
print(f'   Subjects for activity 0 in [s1,s2]: '
      f'{get_subjects_for_activity(0, demo_map, ["s1","s2"])}')

✅ Subject-activity map builder OK
   Demo map: {'s1': [0, 1], 's2': [1, 2], 's3': [0, 2, 3]}
   Subjects for activity 0 in [s1,s2]: ['s1']


## Cell 15 — Experiment Registry
Tracks which configurations have been run. Prevents duplicate runs and enables safe resume.

In [15]:
REGISTRY_PATH = os.path.join(RESULTS_BASE, 'experiment_registry.json')
RESULTS_CSV   = os.path.join(RESULTS_BASE, 'all_results.csv')


def make_config_key(
    dataset:  str,
    backbone: str,
    method:   str,
    k_shot:   int,
    seed:     int
) -> str:
    """Unique string key for one experimental run."""
    return f'{dataset}__{backbone}__{method}__k{k_shot}__seed{seed}'


def load_registry() -> Dict:
    """Load the experiment registry from disk."""
    if os.path.exists(REGISTRY_PATH):
        with open(REGISTRY_PATH) as f:
            return json.load(f)
    return {}


def mark_completed(
    config_key:  str,
    result_dict: Dict
) -> None:
    """Mark a config as completed in the registry."""
    registry = load_registry()
    registry[config_key] = {'status': 'completed', **result_dict}
    with open(REGISTRY_PATH, 'w') as f:
        json.dump(registry, f, indent=2)

def is_completed(config_key: str) -> bool:
    """Check if a configuration has already been run."""
    return config_key in load_registry()


def print_progress() -> None:
    """Print a summary of completed vs total configurations."""
    registry = load_registry()
    total    = len(ALL_DATASETS) * len(ALL_BACKBONES) * len(ALL_FSL_METHODS) * len(K_SHOTS) * len(EVAL_SEEDS)
    done     = len(registry)
    print(f'Progress: {done}/{total} runs completed ({100*done/total:.1f}%)')
    if done > 0:
        datasets_done = set(k.split('__')[0] for k in registry)
        print(f'Datasets touched: {datasets_done}')


print('✅ Experiment registry OK')
print_progress()

✅ Experiment registry OK
Progress: 144/720 runs completed (20.0%)
Datasets touched: {'cogage_atomic', 'humcare', 'wisdm', 'cogage_composite'}


## Cell 16 — Final Verification
Run this cell to confirm everything is correctly set up.

In [16]:
print('=' * 60)
print('  NOTEBOOK 00 — FINAL VERIFICATION')
print('=' * 60)

checks = {
    'seed_everything'          : seed_everything,
    'DATASET_CONFIGS'          : DATASET_CONFIGS,
    'RAW_PATHS'                : RAW_PATHS,
    'PROCESSED_PATHS'          : PROCESSED_PATHS,
    'split_subjects'           : split_subjects,
    'get_split_mask'           : get_split_mask,
    'resample_stream'          : resample_stream,
    'sliding_window'           : sliding_window,
    'StreamNormalizer'         : StreamNormalizer,
    'EpisodeSampler'           : EpisodeSampler,
    'save_processed_dataset'   : save_processed_dataset,
    'load_processed_dataset'   : load_processed_dataset,
    'save_checkpoint'          : save_checkpoint,
    'load_checkpoint'          : load_checkpoint,
    'compute_accuracy'         : compute_accuracy,
    'compute_macro_f1'         : compute_macro_f1,
    'compute_personalization_gain': compute_personalization_gain,
    'summarise_results'        : summarise_results,
    'build_subject_activity_map': build_subject_activity_map,
    'make_config_key'          : make_config_key,
    'is_completed'             : is_completed,
    'mark_completed'           : mark_completed,
    'plot_k_shot_curves'       : plot_k_shot_curves,
    'plot_personalization_gain_heatmap': plot_personalization_gain_heatmap,
    'DEVICE'                   : DEVICE,
    'TARGET_FREQ'              : TARGET_FREQ,
    'WINDOW_SIZE'              : WINDOW_SIZE,
    'EMBEDDING_DIM'            : EMBEDDING_DIM,
    'K_SHOTS'                  : K_SHOTS,
    'EVAL_SEEDS'               : EVAL_SEEDS,
}

all_ok = True
for name, obj in checks.items():
    status = '✅' if obj is not None else '❌'
    if obj is None:
        all_ok = False
    print(f'  {status} {name}')

print('=' * 60)
if all_ok:
    print('  ✅ ALL CHECKS PASSED — ready for notebook 01')
else:
    print('  ❌ SOME CHECKS FAILED — review above')
print('=' * 60)

  NOTEBOOK 00 — FINAL VERIFICATION
  ✅ seed_everything
  ✅ DATASET_CONFIGS
  ✅ RAW_PATHS
  ✅ PROCESSED_PATHS
  ✅ split_subjects
  ✅ get_split_mask
  ✅ resample_stream
  ✅ sliding_window
  ✅ StreamNormalizer
  ✅ EpisodeSampler
  ✅ save_processed_dataset
  ✅ load_processed_dataset
  ✅ save_checkpoint
  ✅ load_checkpoint
  ✅ compute_accuracy
  ✅ compute_macro_f1
  ✅ compute_personalization_gain
  ✅ summarise_results
  ✅ build_subject_activity_map
  ✅ make_config_key
  ✅ is_completed
  ✅ mark_completed
  ✅ plot_k_shot_curves
  ✅ plot_personalization_gain_heatmap
  ✅ DEVICE
  ✅ TARGET_FREQ
  ✅ WINDOW_SIZE
  ✅ EMBEDDING_DIM
  ✅ K_SHOTS
  ✅ EVAL_SEEDS
  ✅ ALL CHECKS PASSED — ready for notebook 01
